# Streams, Tasks & Dynamic Tables

## Test Cases
| Feature | Test Scenario | Target |
|---------|--------------|--------|
| Streams/CDC | Create stream, capture 10K changes, consume via Task | GA |
| Dynamic Tables | Create DT with 1-min lag from Iceberg source | Verify incremental refresh |
| Snowpipe Streaming | Stream 100K JSON records | P99 latency <10s |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE SCHEMA TESTS;
USE WAREHOUSE COMPUTE_WH;

## Test 1: Streams & CDC
Create stream on Iceberg table and capture changes.

In [ ]:
-- Create source Iceberg table for CDC
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.CDC_SOURCE (
    id INT,
    customer_name STRING,
    email STRING,
    status STRING,
    updated_at TIMESTAMP_NTZ(9)
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'EXVOL'
ICEBERG_VERSION = 3;

-- Create stream on Iceberg table
CREATE OR REPLACE STREAM ICEBERG_POC.TESTS.CDC_STREAM ON ICEBERG TABLE ICEBERG_POC.TESTS.CDC_SOURCE;

-- Create target table for captured changes
CREATE OR REPLACE TABLE ICEBERG_POC.TESTS.CDC_TARGET (
    id INT,
    customer_name STRING,
    email STRING,
    status STRING,
    updated_at TIMESTAMP_NTZ(9),
    change_type STRING,
    captured_at TIMESTAMP_NTZ
);

In [ ]:
-- Create Task to consume stream
CREATE OR REPLACE TASK ICEBERG_POC.TESTS.CDC_TASK
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = '1 MINUTE'
    WHEN SYSTEM$STREAM_HAS_DATA('ICEBERG_POC.TESTS.CDC_STREAM')
AS
INSERT INTO ICEBERG_POC.TESTS.CDC_TARGET
SELECT 
    id, customer_name, email, status, updated_at,
    CASE 
        WHEN METADATA$ACTION = 'INSERT' THEN 'INSERT'
        WHEN METADATA$ACTION = 'DELETE' AND METADATA$ISUPDATE THEN 'UPDATE'
        ELSE 'DELETE'
    END AS change_type,
    CURRENT_TIMESTAMP() AS captured_at
FROM ICEBERG_POC.TESTS.CDC_STREAM;

-- Resume task
ALTER TASK ICEBERG_POC.TESTS.CDC_TASK RESUME;

In [ ]:
-- Insert test data to trigger CDC
INSERT INTO ICEBERG_POC.TESTS.CDC_SOURCE
SELECT 
    SEQ4() AS id,
    'Customer_' || SEQ4() AS customer_name,
    'customer' || SEQ4() || '@test.com' AS email,
    'active' AS status,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ(9) AS updated_at
FROM TABLE(GENERATOR(ROWCOUNT => 10000));

-- Check stream has data
SELECT SYSTEM$STREAM_HAS_DATA('ICEBERG_POC.TESTS.CDC_STREAM') AS has_data;
SELECT COUNT(*) AS stream_rows FROM ICEBERG_POC.TESTS.CDC_STREAM;

## Test 2: Dynamic Tables
Create Dynamic Table with 1-min lag from Iceberg source.

In [ ]:
-- Create Dynamic Table from Iceberg source
CREATE OR REPLACE DYNAMIC TABLE ICEBERG_POC.TESTS.EVENTS_SUMMARY
    TARGET_LAG = '1 minute'
    WAREHOUSE = COMPUTE_WH
AS
SELECT 
    region,
    event_type,
    DATE_TRUNC('hour', event_timestamp) AS event_hour,
    COUNT(*) AS event_count,
    SUM(amount) AS total_amount,
    AVG(amount) AS avg_amount
FROM ICEBERG_POC.TESTS.EVENTS
GROUP BY 1, 2, 3;

-- Check Dynamic Table status
SHOW DYNAMIC TABLES LIKE 'EVENTS_SUMMARY' IN SCHEMA ICEBERG_POC.TESTS;

In [ ]:
-- Insert new data to verify incremental refresh
INSERT INTO ICEBERG_POC.TESTS.EVENTS
SELECT 
    (SELECT MAX(event_id) FROM ICEBERG_POC.TESTS.EVENTS) + SEQ4() AS event_id,
    UNIFORM(1, 100000, RANDOM()) AS customer_id,
    'new_event' AS event_type,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ(9) AS event_timestamp,
    PARSE_JSON('{"source": "dt_test"}') AS event_data,
    'US-EAST' AS region,
    ROUND(UNIFORM(100, 5000, RANDOM()) / 100.0, 2) AS amount
FROM TABLE(GENERATOR(ROWCOUNT => 100000));

-- Check Dynamic Table refresh
SELECT * FROM ICEBERG_POC.TESTS.EVENTS_SUMMARY WHERE event_type = 'new_event';